# Air Traffic Flow Management 
To avoid congestion in critical air sectors, the take-off of flights is delayed.
* Limitations of air capacity are expressed in terms of regulated periods, i.e. an interval with an hourly capacity rate.  
* Regulated air-traffic sectors have one or several regulated periods, i.e. their capacity is limited during these periods.
* The flights have an expected take-off time (ETOT) which is specified in hours, minutes, and seconds - converted into total number of minutes.
* An enter event specifies that a given flight will enter a given sector at an expected time (called expected time over).

Objective is to minimize the total sum of flight delays.

In [26]:
import docplex.mp
from math import sqrt
import random

In [ ]:
numFlights = 5
flights = range(numFlights)
sectorNames = []

# times are specified in hours, minutes, seconds 
# (in general year, month, day are needed as well)
tuple Time {
    int hours;
    int minutes;
    int seconds;
};

# limitations of air capacity are expressed in terms of regulated periods, i.e. an interval with an hourly capacity rate
tuple Period {
    Time start;
    Time end;
    int rate;
};

{Period} periods[SectorNames] = ...;

# an enter event specifies that a given flight will enter a given sector at an expected time (called expected time over)
tuple Enter { 
    int flight;
    string sector;
    Time eto;
};

numEnters = 5
enters = range(numEnters)
Enter e[Enters] = ...;

# flight delays will be limited to 2 hours
maxDelay = 120

# capacity of the resource will be made available by time steps of 10 minutes
timeStep = 10

# flight delays are expressed by integer variables
dvar int delay[Flights] in 0 .. maxDelay;

# each enter event is modelled by an activity of duration 1
dvar interval a[Enters] size 1;

# each sector is modelled by a resource
cumulFunction r[i in SectorNames] = sum(en in Enters : e[en].sector == i) pulse(a[en], 1);

execute {
                 cp.param.FailLimit = 20000;
}

dexpr int totalDelay = sum(i in Flights) delay[i];
 
minimize totalDelay;
constraints {

    # the capacity rate is adapted to intervals of 10 minutes;
    # the time scale of a resource is divided by the time step
   forall (i in SectorNames)
       forall (p in periods[i])
          alwaysIn(r[i], (p.start.hours * 60 + p.start.minutes) div timeStep,
                         (p.end.hours * 60 + p.end.minutes) div timeStep,
                            0,
                         (p.rate * timeStep + 59) div 60);


        // a flight enters a sector at its expected time-over plus its delay;
        // since the time scale of a resource is divided by the time step,
        // we do the same for the start time of the activity
        forall (i in Enters)
           startOf(a[i]) == (delay[e[i].flight] + e[i].eto.hours * 60 + e[i].eto.minutes) div timeStep;

            forall(i in SectorNames)
          r[i] <= nbOfFlights;
     }
 
    execute {
       writeln("total delay = " + totalDelay);
    }

In [ ]:
using JuMP, AmplNLWriter
# 2D NonConvex Aircraft Conflict Resolution Problem (ARCP)

#d = 0.05 # 5
d = 0.5 # 10

radius = 2.0

#cap = [1.0, 1.2, 1.4, 1.6, 1.8, 0, 0.2, 0.4, 0.6, 0.8] * pi
cap = [3.14159, 3.76991, 4.39823, 5.02655, 5.65487, 0.00000, 0.62832, 1.25664, 1.88496, 2.51327]

#cap = [1.0, 1.4, 1.8, 0.2, 0.6] * 3.14159
#cap = [3.14159, 4.39823, 5.65487, 0.62832, 1.88496]

x0 = [2.0, 1.62, 0.62, -0.62, -1.62, -2.0, -1.62, -0.62, 0.62, 1.62]
#x0 = [2.0, 0.62, -1.62, -1.62, 0.62]

y0 = [-0.0, 1.18, 1.90, 1.90, 1.18, 0.0, -1.18, -1.90, -1.90, -1.18]
#y0 = [0, 1.90, 1.18, -1.18, -1.90]

n = length(cap)
v0 = repeat([5.0],n)

A = collect(1:n)
wd = 0.5

eps = 1e-3
M = 1e4

theta0 = [val >= pi ? val - 2*pi : val for val in cap]

xr0 = [x0[i] - x0[j] for i in A, j in A]

yr0 = [y0[i] - y0[j] for i in A, j in A]

ucc = 1

hmin = -pi / 6
hmax = pi / 6

lcc = min(cos(hmin),cos(hmax))
uss = sin(hmax)
lss = sin(hmin)

# control bounds
qmin = 0.94
qmax = 1.03

uvrx = Array{Union{Float64,Missing},2}(undef,n, n)
lvrx = Array{Union{Float64,Missing},2}(undef,n, n)
uvry = Array{Union{Float64,Missing},2}(undef,n, n)
lvry = Array{Union{Float64,Missing},2}(undef,n, n)
a = Array{Union{Float64,Missing},2}(undef,n, n)
b = Array{Union{Float64,Missing},2}(undef,n, n)
c = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin11 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin12 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin311 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin312 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin321 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin322 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin331 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin332 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin341 = Array{Union{Float64,Missing},2}(undef,n, n)
Mbin342 = Array{Union{Float64,Missing},2}(undef,n, n)
gammal = Array{Union{Float64,Missing},2}(undef,n, n)
gammau = Array{Union{Float64,Missing},2}(undef,n, n)
phil = Array{Union{Float64,Missing},2}(undef,n, n)
phiu = Array{Union{Float64,Missing},2}(undef,n, n)

for (i,j) in [(i,j) for i in A, j in A if i < j]
  if cos(theta0[i])>=0 && cos(theta0[j])>=0 && sin(theta0[i])>=0 && sin(theta0[j])>=0 # 1
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])>=0 && sin(theta0[i])>=0 && sin(theta0[j])< 0 # 2
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])>=0 && sin(theta0[i])< 0 && sin(theta0[j])>=0 # 3
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])>=0 && sin(theta0[i])< 0 && sin(theta0[j])< 0 # 4
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])< 0 && sin(theta0[i])>=0 && sin(theta0[j])>=0 # 5
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])< 0 && sin(theta0[i])>=0 && sin(theta0[j])< 0 # 6
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])< 0 && sin(theta0[i])< 0 && sin(theta0[j])>=0 # 7
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])>=0 && cos(theta0[j])< 0 && sin(theta0[i])< 0 && sin(theta0[j])< 0 # 8
    uvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])>=0 && sin(theta0[i])>=0 && sin(theta0[j])>=0 # 9
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])>=0 && sin(theta0[i])>=0 && sin(theta0[j])< 0 # 10
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])>=0 && sin(theta0[i])< 0 && sin(theta0[j])>=0 # 11
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])>=0 && sin(theta0[i])< 0 && sin(theta0[j])< 0 # 12
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])< 0 && sin(theta0[i])>=0 && sin(theta0[j])>=0 # 13
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])< 0 && sin(theta0[i])>=0 && sin(theta0[j])< 0 # 14
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])< 0 && sin(theta0[i])< 0 && sin(theta0[j])>=0 # 15
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
  end
  if cos(theta0[i])< 0 && cos(theta0[j])< 0 && sin(theta0[i])< 0 && sin(theta0[j])< 0 # 16
    uvrx[i,j] = qmin*lcc*v0[i]*cos(theta0[i]) - qmax*uss*v0[i]*sin(theta0[i]) - qmax*ucc*v0[j]*cos(theta0[j]) + qmin*lss*v0[j]*sin(theta0[j])
    lvrx[i,j] = qmax*ucc*v0[i]*cos(theta0[i]) - qmin*lss*v0[i]*sin(theta0[i]) - qmin*lcc*v0[j]*cos(theta0[j]) + qmax*uss*v0[j]*sin(theta0[j])
    uvry[i,j] = qmin*lss*v0[i]*cos(theta0[i]) + qmin*lcc*v0[i]*sin(theta0[i]) - qmax*uss*v0[j]*cos(theta0[j]) - qmax*ucc*v0[j]*sin(theta0[j])
    lvry[i,j] = qmax*uss*v0[i]*cos(theta0[i]) + qmax*ucc*v0[i]*sin(theta0[i]) - qmin*lss*v0[j]*cos(theta0[j]) - qmin*lcc*v0[j]*sin(theta0[j])
  end

  a[i,j] = yr0[i,j]^2 - d^2
  b[i,j] = xr0[i,j]^2 - d^2
  c[i,j] = 2*xr0[i,j]*yr0[i,j]
	
  if (xr0[i,j]>=0 && yr0[i,j]>=0) # 1
    Mbin11[i,j] = xr0[i,j]*uvry[i,j] - yr0[i,j]*lvrx[i,j]
    Mbin12[i,j] = -xr0[i,j]*lvry[i,j] + yr0[i,j]*uvrx[i,j]
    if (b[i,j]>=0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin331[i,j] = 2*b[i,j]*uvry[i,j] - uvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (b[i,j]>=0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin331[i,j] = 2*b[i,j]*uvry[i,j] - lvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end	
    if (b[i,j]< 0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin331[i,j] = 2*b[i,j]*lvry[i,j] - uvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (b[i,j]< 0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin331[i,j] = 2*b[i,j]*lvry[i,j] - lvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (a[i,j]>=0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin332[i,j] = 2*a[i,j]*uvrx[i,j] - uvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (a[i,j]>=0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin332[i,j] = 2*a[i,j]*uvrx[i,j] - lvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end		
    if (a[i,j]< 0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin332[i,j] = 2*a[i,j]*lvrx[i,j] - uvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (a[i,j]< 0 && -(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin332[i,j] = 2*a[i,j]*lvrx[i,j] - lvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    gammal[i,j] = 2*b[i,j]
    gammau[i,j] = 2*a[i,j]
    phil[i,j] = c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
    phiu[i,j] = c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
  end
  if (xr0[i,j]>=0 && yr0[i,j]< 0) # 2
    Mbin11[i,j] = xr0[i,j]*uvry[i,j] - yr0[i,j]*uvrx[i,j]
    Mbin12[i,j] = -xr0[i,j]*lvry[i,j] + yr0[i,j]*lvrx[i,j]		
    if (a[i,j]>=0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin311[i,j] = 2*a[i,j]*uvrx[i,j] - uvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (a[i,j]>=0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin311[i,j] = 2*a[i,j]*uvrx[i,j] - lvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end			
    if (a[i,j]< 0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin311[i,j] = 2*a[i,j]*lvrx[i,j] - uvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (a[i,j]< 0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin311[i,j] = 2*a[i,j]*lvrx[i,j] - lvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end	
    if (-b[i,j]>=0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin312[i,j] = -2*b[i,j]*uvry[i,j] + uvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-b[i,j]>=0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin312[i,j] = -2*b[i,j]*uvry[i,j] + lvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-b[i,j]< 0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin312[i,j] = -2*b[i,j]*lvry[i,j] + uvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-b[i,j]< 0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin312[i,j] = -2*b[i,j]*lvry[i,j] + lvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    gammal[i,j] = 2*a[i,j]
    gammau[i,j] = -2*b[i,j]
    phil[i,j] = c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
    phiu[i,j] = c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
  end
  if (xr0[i,j]< 0 && yr0[i,j]>=0) # 3
    Mbin11[i,j] = xr0[i,j]*lvry[i,j] - yr0[i,j]*lvrx[i,j]
    Mbin12[i,j] = -xr0[i,j]*uvry[i,j] + yr0[i,j]*uvrx[i,j]
    if (-a[i,j]>=0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin321[i,j] = -2*a[i,j]*uvrx[i,j] + uvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-a[i,j]>=0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin321[i,j] = -2*a[i,j]*uvrx[i,j] + lvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end		
    if (-a[i,j]< 0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin321[i,j] = -2*a[i,j]*lvrx[i,j] + uvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-a[i,j]< 0 && (c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin321[i,j] = -2*a[i,j]*lvrx[i,j] + lvry[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (b[i,j]>=0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin322[i,j] = 2*b[i,j]*uvry[i,j] - uvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (b[i,j]>=0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin322[i,j] = 2*b[i,j]*uvry[i,j] - lvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end			
    if (b[i,j]< 0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin322[i,j] = 2*b[i,j]*lvry[i,j] - uvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (b[i,j]< 0 && -(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin322[i,j] = 2*b[i,j]*lvry[i,j] - lvrx[i,j]*(c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    gammal[i,j] = -2*a[i,j]
    gammau[i,j] = 2*b[i,j]
    phil[i,j] = c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
    phiu[i,j] = c[i,j] - sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
  end
  if (xr0[i,j]< 0 && yr0[i,j]< 0) # 4
    Mbin11[i,j] = xr0[i,j]*lvry[i,j] - yr0[i,j]*uvrx[i,j]
    Mbin12[i,j] = -xr0[i,j]*uvry[i,j] + yr0[i,j]*lvrx[i,j]
    if (-b[i,j]>=0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin341[i,j] = -2*b[i,j]*uvry[i,j] + uvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-b[i,j]>=0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin341[i,j] = -2*b[i,j]*uvry[i,j] + lvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end	
    if (-b[i,j]< 0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin341[i,j] = -2*b[i,j]*lvry[i,j] + uvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-b[i,j]< 0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin341[i,j] = -2*b[i,j]*lvry[i,j] + lvrx[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end		
    if (-a[i,j]>=0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin342[i,j] = -2*a[i,j]*uvrx[i,j] + uvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-a[i,j]>=0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin342[i,j] = -2*a[i,j]*uvrx[i,j] + lvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-a[i,j]< 0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))>=0) Mbin342[i,j] = -2*a[i,j]*lvrx[i,j] + uvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    if (-a[i,j]< 0 && (c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j]))< 0) Mbin342[i,j] = -2*a[i,j]*lvrx[i,j] + lvry[i,j]*(c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])) end
    gammal[i,j] = -2*b[i,j]
    gammau[i,j] = -2*a[i,j]
    phil[i,j] = c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
    phiu[i,j] = c[i,j] + sqrt(c[i,j]^2 - 4*a[i,j]*b[i,j])
  end
end

#Opt = optimizer_with_attributes(() -> AmplNLWriter.Optimizer("/opt/Couenne/bin/couenne"))
Opt = optimizer_with_attributes(() -> AmplNLWriter.Optimizer("/home/mgd/packages/bin/bonmin"))
#Opt = optimizer_with_attributes(() -> AmplNLWriter.Optimizer("./scip"))
model = Model(Opt)
@variable(model, qmin <= q[I=A] <= qmax)
@variable(model, hmin <= theta[I=A] <= hmax)
@variable(model, vrx[I=A,J=A; I < J])
@variable(model, vry[I=A,J=A; I < J])
@variable(model, z[I=A,J=A; I<J], Bin)

@constraint(model, cvrx_range[I=A,J=A; I < J], lvrx[I,J] <= vrx[I,J] <= uvrx[I,J])
@constraint(model, cvry_range[I=A,J=A; I < J], lvry[I,J] <= vry[I,J] <= uvry[I,J])
@NLconstraint(model, cvrx[I=A,J=A; I < J], vrx[I,J] == q[I]*v0[I]*cos(theta[I] + theta0[I]) - q[J]*v0[J]*cos(theta[J]+theta0[J]))
@NLconstraint(model, cvry[I=A,J=A; I < J], vry[I,J] == q[I]*v0[I]*sin(theta[I] + theta0[I]) - q[J]*v0[J]*sin(theta[J]+theta0[J]))
@constraint(model, bin11[I=A,J=A; I < J], xr0[I,J]*vry[I,J] - yr0[I,J]*vrx[I,J] <= (1 - z[I,J])*Mbin11[I,J])
@constraint(model, bin12[I=A,J=A; I < J], xr0[I,J]*vry[I,J] - yr0[I,J]*vrx[I,J] >= -(z[I,J])*Mbin12[I,J])
@constraint(model, bin311[I=A,J=A; I < J && xr0[I,J] >= 0 && yr0[I,J] < 0], gammal[I,J]*vrx[I,J] - phil[I,J]*vry[I,J] <= (1 - z[I,J])*Mbin311[I,J])
@constraint(model, bin312[I=A,J=A; I < J && xr0[I,J] >= 0 && yr0[I,J] < 0], gammau[I,J]*vry[I,J] + phiu[I,J]*vrx[I,J] <= z[I,J]*Mbin312[I,J])
@constraint(model, bin321[I=A,J=A; I < J && xr0[I,J] <  0 && yr0[I,J] >=0], gammal[I,J]*vrx[I,J] + phil[I,J]*vry[I,J] <= (1 - z[I,J])*Mbin321[I,J])
@constraint(model, bin322[I=A,J=A; I < J && xr0[I,J] <  0 && yr0[I,J] >=0], gammau[I,J]*vry[I,J] - phiu[I,J]*vrx[I,J] <= z[I,J]*Mbin322[I,J])
@constraint(model, bin331[I=A,J=A; I < J && xr0[I,J] >= 0 && yr0[I,J] >=0], gammal[I,J]*vry[I,J] - phil[I,J]*vrx[I,J] <= (1 - z[I,J])*Mbin331[I,J])
@constraint(model, bin332[I=A,J=A; I < J && xr0[I,J] >= 0 && yr0[I,J] >=0], gammau[I,J]*vrx[I,J] - phiu[I,J]*vry[I,J] <= z[I,J]*Mbin332[I,J])
@constraint(model, bin341[I=A,J=A; I < J && xr0[I,J] <  0 && yr0[I,J] < 0], gammal[I,J]*vry[I,J] + phil[I,J]*vrx[I,J] <= (1 - z[I,J])*Mbin341[I,J])
@constraint(model, bin342[I=A,J=A; I < J && xr0[I,J] <  0 && yr0[I,J] < 0], gammau[I,J]*vrx[I,J] + phiu[I,J]*vry[I,J] <= z[I,J]*Mbin342[I,J])

@objective(model,Min,sum(wd*theta[I]^2 + (1-wd)*(1-q[I])^2 for I in A))

optimize!(model)


In [27]:
# Setup Data
N = 3 #8
box_range = range(1, N+1)
obj_range = range(1, N+1)

o_xmax = N*10
o_ymax = 2*N
box_coords = {b: (10*b, 1) for b in box_range}

obj_coords= {1: (140, 6), 2: (146, 8), 3: (132, 14), 4: (53, 28), 
             5: (146, 4), 6: (137, 13), 7: (95, 12), 8: (68, 9), 9: (102, 18), 
             10: (116, 8), 11: (19, 29), 12: (89, 15), 13: (141, 4), 14: (29, 4), 15: (4, 28)}

# the distance matrix from box i to object j - we compute the square of distance to keep integer
distances = {}
for o in obj_range:
    for b in box_range:
        dx = obj_coords[o][0]-box_coords[b][0]
        dy = obj_coords[o][1]-box_coords[b][1]
        d2 = dx*dx + dy*dy
        distances[b, o] = d2

In [28]:
# Create the Model
from docplex.mp.model import Model
mdl = Model("boxes")

#### Define the decision variables

* For each box $i$ ($i$ in $1..N$) and object $j$ ($j$ in $1..N$), we define a binary variable $X_{i,j}$ equal to $1$ if and only if object $j$ is stored in box $i$.

In [29]:
# decision variables is a 2d-matrix
x = mdl.binary_var_matrix(box_range, obj_range, lambda ij: "x_%d_%d" %(ij[0], ij[1]))

#### Express the business constraints

* The sum of $X_{i,j}$ over both rows and columns must be equal to $1$, resulting in $2\times N$ constraints.

In [30]:
# one object per box
mdl.add_constraints(mdl.sum(x[i,j] for j in obj_range) == 1
                   for i in box_range)
    
# one box for each object
mdl.add_constraints(mdl.sum(x[i,j] for i in box_range) == 1
                  for j in obj_range)

mdl.print_information()

Model: boxes
 - number of variables: 9
   - binary=9, integer=0, continuous=0
 - number of constraints: 6
   - linear=6
 - parameters: defaults
 - objective: none
 - problem type is: MILP


#### Express the objective

* The objective is to minimize the total distance between each object and its storage box.

In [31]:
# minimize total displacement
mdl.minimize( mdl.sum(distances[i,j] * x[i,j] for i in box_range for j in obj_range) )

In [32]:
mdl.export_as_lp(path='800-Objects in Boxes.lp')

'800-Objects in Boxes.lp'

#### Solve the model


In [33]:
mdl.print_information()

assert mdl.solve(), "!!! Solve of the model fails"

Model: boxes
 - number of variables: 9
   - binary=9, integer=0, continuous=0
 - number of constraints: 6
   - linear=6
 - parameters: defaults
 - objective: minimize
 - problem type is: MILP


In [34]:
mdl.report()
d1 = mdl.objective_value
#mdl.print_solution()

def make_solution_vector(x_vars):
    sol = [0]* N
    for i in box_range:
        for j in obj_range:
            if x[i,j].solution_value >= 0.5:
                sol[i-1] = j
                break
    return sol

def make_obj_box_dir(sol_vec):
    # sol_vec contains an array of objects in box order at slot b-1 we have obj(b)
    return { sol_vec[b]: b+1 for b in range(N)}
    
               
sol1 = make_solution_vector(x)
print("* solution: {0!s}".format(sol1))          

* model boxes solved with objective = 42983.000
* solution: [3, 1, 2]
